In [3]:
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag,ResultSet
import random
import pandas as pd
import time
import re
from datatypes import JobPosting
from typing import List
from enum import Enum
from dataclasses import dataclass


In [4]:


class FilterEnum(Enum):
    # This method can now be used by any Enum inheriting from ParamEnum
    @classmethod
    def get_param(cls,value, param_name):
        for item in cls:
            if item.value == value:
                return f"{param_name}={item.value}" if item.value is not None else ""
        return ""


class FilterTime(FilterEnum):
    PAST_24_HOURS = 'r86400'
    PAST_WEEK = 'r604800'
    PAST_MONTH = 'r2592000'
    ANY_TIME = None
    
    # Specific implementation for TimeFilter
    @classmethod
    def get_param(cls, value):
        return super().get_param(value, "f_TPR")

class FilterSalaryRange(FilterEnum):
    RANGE_40K_PLUS = 1
    RANGE_60K_PLUS = 2
    RANGE_80K_PLUS = 3
    RANGE_100K_PLUS = 4
    RANGE_120K_PLUS = 5
    RANGE_140K_PLUS = 6
    RANGE_160K_PLUS = 7
    RANGE_180K_PLUS = 8
    RANGE_200K_PLUS = 9
    RANGE_ANY = None

    @classmethod
    def get_param(cls, value):
        return super().get_param(value, "f_SB2")

class FilterExperienceLevel(FilterEnum):
    INTERNSHIP = 1
    ENTRY_LEVEL = 2
    ASSOCIATE = 3
    MID_SENIOR_LEVEL = 4
    DIRECTOR = 5
    EXECUTIVE = 6
    ANY_EXPERIENCE_LEVEL = None

    @classmethod
    def get_param(cls, value):
        return super().get_param(value, "f_E")

def build_company_search_url(company_id:int = None):
    return f"f_C={company_id}" if company_id is not None else ""

class FilterRemoteModality(FilterEnum):
    ON_SITE = 1
    REMOTE = 2
    HYBRID =3
    ANY_MODALITY = None

    @classmethod
    def get_param(cls, value):
        return super().get_param(value, "f_WT")


def parse_input(query_parameter:str,
                input_str:str,):
    input_str = input_str.strip()
    input_str = input_str.lower()
    input_str = input_str.replace(" ","%20")
    return f"{query_parameter}={input_str}"

In [5]:
base_url = url = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"

# Search bar
keywords_input= "data scientist"
url = base_url + parse_input("keywords",keywords_input)
# Location
location_input="Washington DC"
url += parse_input("location",location_input)

# Particular company id 
#company_id =10667 # Meta
company_id = None
url += build_company_search_url(company_id)


# Salary range
salary = FilterSalaryRange.RANGE_100K_PLUS
if salary is not None:
    url += salary.get_param(salary.value)

time_filter = FilterTime.PAST_MONTH
url += time_filter.get_param(time_filter.value)

experience_level = FilterExperienceLevel.MID_SENIOR_LEVEL
url += experience_level.get_param(experience_level.value)

remote_modality = FilterRemoteModality.ON_SITE
url += remote_modality.get_param(remote_modality.value)

url


'https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/searchkeywords=data%20scientistlocation=washington%20dcf_SB2=4f_TPR=r2592000f_E=4f_WT=1'